# Compare K-means candidates: k=1 through k=8

Run this notebook in Google Colab with **Runtime > Run all**. It is self-contained and uses the same frozen input-data commit as environment_check.ipynb.

The goal is to compare eight candidate models and inspect their training and test elbow curves before choosing k. All models use the same six scaled features and the same 231/99 split. Each model is fitted on training members only.

The notebook exports charts, numerical results, every member's cluster and squared distance, and every fitted center. It leaves the choice of k and business interpretation to the student.

## 1. Prepare the runtime

This repeats the package and data checks from the successful environment notebook, so this notebook can run in a fresh Colab session.

In [ ]:
%pip install -q numpy scipy pandas scikit-learn matplotlib

In [ ]:
import sys
import numpy as np
import numpy.random
import scipy
import pandas as pd
import sklearn
import matplotlib
from sklearn.cluster import KMeans

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Ready for K-means')
import json
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
DATA_REF = "24cd0a915b7a3ee05f6a4635190d05130385f25b"
BASE_URL = f"https://raw.githubusercontent.com/FrankZhao111/ClusterHW/{DATA_REF}/"
FEATURES = [
    "age",
    "visits_per_month",
    "avg_visit_minutes",
    "classes_per_month",
    "months_member",
    "monthly_extras_spend",
]

train = pd.read_csv(BASE_URL + "train_scaled.csv")
test = pd.read_csv(BASE_URL + "test_scaled.csv")
raw_train = pd.read_csv(BASE_URL + "train_members.csv")
raw_test = pd.read_csv(BASE_URL + "test_members.csv")
parameters = pd.read_csv(BASE_URL + "scaling_parameters.csv").set_index("feature").loc[FEATURES]

assert list(train.columns) == ["member_id", *FEATURES]
assert list(test.columns) == ["member_id", *FEATURES]
assert train.shape == (231, 7) and test.shape == (99, 7)
assert train["member_id"].is_unique and test["member_id"].is_unique
assert not set(train["member_id"]) & set(test["member_id"])
assert (parameters["training_count"] == 231).all()
assert (parameters["ddof"] == 0).all()

X_train = train.loc[:, FEATURES].to_numpy(dtype=float)
X_test = test.loc[:, FEATURES].to_numpy(dtype=float)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
np.testing.assert_allclose(X_train.mean(axis=0), 0, atol=1e-10)
np.testing.assert_allclose(X_train.std(axis=0, ddof=0), 1, atol=1e-10)

# Align raw rows by member ID to check every transformed value.
for raw, scaled in [(raw_train, train), (raw_test, test)]:
    aligned = raw.set_index("member_id").loc[scaled["member_id"], FEATURES]
    expected = (aligned - parameters["training_mean"]) / parameters["training_std"]
    np.testing.assert_allclose(
        expected.to_numpy(dtype=float),
        scaled.loc[:, FEATURES].to_numpy(dtype=float),
        rtol=1e-12,
        atol=1e-12,
    )

print('Data checks passed: 231 training members, 99 test members, six features.')
print('GitHub data commit:', DATA_REF)

## 2. Fit the candidate models

For each k, initialize and fit K-means on X_train, then assign X_test to the resulting centers. Test members never update centers.

- n_clusters=k is the number of groups.
- n_init=20 tries 20 initializations and keeps the solution with the smallest training spread.
- random_state=42 fixes the initialization randomness for reproducibility.
- Spread is the sum, across members, of their squared Euclidean distance to their assigned center in the six standardized features.

The code also calculates member-level squared distances and verifies their sums against scikit-learn's inertia and score.

In the results table, a drop is previous-k spread minus current-k spread. The first drop is blank because k=1 has no preceding candidate. A reduction versus k=1 is (baseline spread - candidate spread) / baseline spread x 100. It is not an accuracy percentage.

In [ ]:
FIT_CONFIG = {
    "init": "k-means++",
    "n_init": 20,
    "random_state": 42,
    "max_iter": 300,
    "tol": 1e-4,
    "algorithm": "lloyd",
}
models = {}
metric_rows = []
assignment_tables = []
scaled_center_rows = []
original_center_rows = []
training_mean = parameters.loc[FEATURES, "training_mean"].to_numpy(dtype=float)
training_std = parameters.loc[FEATURES, "training_std"].to_numpy(dtype=float)

for k in range(1, 9):
    model = KMeans(n_clusters=k, **FIT_CONFIG)
    model.fit(X_train)  # Every fit uses ONLY the training members.
    models[k] = model

    train_labels = model.predict(X_train)
    test_labels = model.predict(X_test)  # Assign to existing centers; no refit.
    train_distances_sq = np.sum(
        (X_train - model.cluster_centers_[train_labels]) ** 2, axis=1
    )
    test_distances_sq = np.sum(
        (X_test - model.cluster_centers_[test_labels]) ** 2, axis=1
    )
    train_spread = float(train_distances_sq.sum())
    test_spread = float(test_distances_sq.sum())

    # Independently confirm the member-level sums match scikit-learn's objective.
    np.testing.assert_allclose(train_spread, model.inertia_, rtol=1e-10, atol=1e-8)
    np.testing.assert_allclose(test_spread, -model.score(X_test), rtol=1e-10, atol=1e-8)
    metric_rows.append({
        "k": k,
        "train_spread": train_spread,
        "test_spread": test_spread,
        "train_spread_per_member": train_spread / len(X_train),
        "test_spread_per_member": test_spread / len(X_test),
        "n_iterations": int(model.n_iter_),
    })

    for split_name, frame, labels, distances_sq in [
        ("train", train, train_labels, train_distances_sq),
        ("test", test, test_labels, test_distances_sq),
    ]:
        assignment_tables.append(pd.DataFrame({
            "k": k,
            "split": split_name,
            "member_id": frame["member_id"].to_numpy(),
            "cluster": labels,
            "squared_distance": distances_sq,
        }))

    original_centers = model.cluster_centers_ * training_std + training_mean
    for cluster in range(k):
        common = {"k": k, "cluster": cluster, "training_members": int(np.sum(train_labels == cluster))}
        scaled_center_rows.append({
            **common,
            **dict(zip(FEATURES, model.cluster_centers_[cluster].tolist())),
        })
        original_center_rows.append({
            **common,
            **dict(zip(FEATURES, original_centers[cluster].tolist())),
        })

    print(f'k={k}: training spread={train_spread:.4f}; test spread={test_spread:.4f}')

metrics = pd.DataFrame(metric_rows)
metrics["train_drop_from_previous"] = metrics["train_spread"].shift(1) - metrics["train_spread"]
metrics["test_drop_from_previous"] = metrics["test_spread"].shift(1) - metrics["test_spread"]
metrics["train_reduction_vs_k1_pct"] = 100 * (1 - metrics["train_spread"] / metrics.loc[0, "train_spread"])
metrics["test_reduction_vs_k1_pct"] = 100 * (1 - metrics["test_spread"] / metrics.loc[0, "test_spread"])
assignments = pd.concat(assignment_tables, ignore_index=True)
centers_scaled = pd.DataFrame(scaled_center_rows)
centers_original = pd.DataFrame(original_center_rows)

# Each of the 330 members appears once per candidate k: 330 x 8 = 2,640 rows.
assert len(assignments) == 330 * 8
assert len(centers_scaled) == sum(range(1, 9)) == 36
grouped = assignments.groupby(["k", "split"])["squared_distance"].sum().unstack()
np.testing.assert_allclose(grouped["train"].to_numpy(), metrics["train_spread"].to_numpy())
np.testing.assert_allclose(grouped["test"].to_numpy(), metrics["test_spread"].to_numpy())
np.testing.assert_allclose(
    metrics.loc[0, ["train_spread", "test_spread"]].to_numpy(dtype=float),
    [1386.0, 666.1365956205666],
    rtol=1e-10,
    atol=1e-10,
)

print('Candidate fitting and member-level spread checks passed.')
display(metrics.round(4))

## 3. Save numerical results and fitted centers

Keep these files for the later interpretation and offer-policy steps. Centers are saved in scaled units and converted back to original units using the existing training parameters.

candidate_assignments.csv has 2,640 rows: 330 members x eight candidate models. Select a single k to obtain one assignment per member. Cluster numbers are identifiers, not rankings.

model_checkpoint.json records the fitted centers, input revision, feature order, scaling parameters, fit configuration, and package versions. It does not contain an automatically chosen k.

In [ ]:
OUTPUT_DIR = Path("outputs/kmeans_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV calculations retain precision; only displayed tables and chart labels are rounded.
metrics.to_csv(OUTPUT_DIR / "fit_metrics.csv", index=False, float_format="%.17g")
assignments.to_csv(OUTPUT_DIR / "candidate_assignments.csv", index=False, float_format="%.17g")
centers_scaled.to_csv(OUTPUT_DIR / "candidate_centers_scaled.csv", index=False, float_format="%.17g")
centers_original.to_csv(OUTPUT_DIR / "candidate_centers_original.csv", index=False, float_format="%.17g")
parameters.reset_index().to_csv(OUTPUT_DIR / "scaling_parameters.csv", index=False, float_format="%.17g")

checkpoint = {
    "repository": "FrankZhao111/ClusterHW",
    "data_commit": DATA_REF,
    "features": FEATURES,
    "training_count": int(len(X_train)),
    "test_count": int(len(X_test)),
    "selected_k": None,
    "kmeans_configuration": FIT_CONFIG,
    "scaling": {
        "mean": training_mean.tolist(),
        "std": training_std.tolist(),
        "ddof": 0,
    },
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "models": {
        str(k): {
            "centers_scaled": models[k].cluster_centers_.tolist(),
            "training_spread": float(models[k].inertia_),
            "n_iterations": int(models[k].n_iter_),
        }
        for k in range(1, 9)
    },
}
(OUTPUT_DIR / "model_checkpoint.json").write_text(
    json.dumps(checkpoint, indent=2) + '\n', encoding='utf-8'
)
print('Saved the comparison table, member assignments, centers, and model checkpoint.')

## 4. Inspect the elbow curves

The first chart shows the required total spreads in adjacent training and test panels. Each point corresponds to the same row of fit_metrics.csv. Chart labels are rounded to one decimal place; CSV values retain precision.

The optional second chart divides each spread by its set's member count. This helps compare magnitudes because training contains 231 members and test contains 99.

Look for where adding another group gives a much smaller improvement. Check whether the test curve supports a similar elbow. Operations prefers three to five segments, but the plotted evidence should guide the decision. If the curves disagree, report the disagreement and reconsider the choice.

Do not select k=8 merely because its training spread is smallest. Do not assume k=3 from the separate lecture example.

In [ ]:
with plt.rc_context({'font.size': 11}):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, column, title, color, marker in [
        (axes[0], 'train_spread', 'Training: 231 members', '#245A81', 'o'),
        (axes[1], 'test_spread', 'Test: 99 members', '#B45F06', 's'),
    ]:
        ax.plot(metrics['k'], metrics[column], color=color, marker=marker, linewidth=2)
        ax.set_title(title)
        ax.set_xlabel('Number of clusters (k)')
        ax.set_ylabel('Sum of squared distances')
        ax.set_xticks(range(1, 9))
        ax.set_ylim(0, float(metrics[column].max()) * 1.20)
        ax.margins(x=0.08)
        ax.grid(alpha=0.25)
        for k_value, spread_value in zip(metrics['k'], metrics[column]):
            ax.annotate(
                f'{spread_value:.1f}', (k_value, spread_value),
                xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8
            )
    fig.suptitle('Training and test spread across candidate cluster counts', fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.savefig(OUTPUT_DIR / 'elbow_train_test.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / 'elbow_train_test.svg', bbox_inches='tight')
    plt.show()

# Optional check: divide by each set's member count to compare magnitudes fairly.
with plt.rc_context({'font.size': 11}):
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    ax.plot(
        metrics['k'], metrics['train_spread_per_member'],
        marker='o', linewidth=2, color='#245A81', label='Training (231 members)'
    )
    ax.plot(
        metrics['k'], metrics['test_spread_per_member'],
        marker='s', linestyle='--', linewidth=2, color='#B45F06', label='Test (99 members)'
    )
    ax.set_title('Spread per member across candidate cluster counts')
    ax.set_xlabel('Number of clusters (k)')
    ax.set_ylabel('Mean squared distance per member')
    ax.set_xticks(range(1, 9))
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'elbow_per_member.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / 'elbow_per_member.svg', bbox_inches='tight')
    plt.show()

## 5. Download and preserve the results

The next cell downloads kmeans_outputs.zip. If the download does not start, open Colab's Files sidebar, find kmeans_outputs.zip, and download it manually.

Extract the ZIP and upload its CSV files, model_checkpoint.json, and charts to your GitHub repository. In particular, share fit_metrics.csv and elbow_train_test.png for the next review. Save the executed notebook to GitHub or download it and upload it to preserve the run outputs.

Then choose and justify k from the training and test evidence. Naming segments and deciding who gets the credit come after that choice. The student writes the final slides.

In [ ]:
archive_path = shutil.make_archive('kmeans_outputs', 'zip', root_dir=OUTPUT_DIR)
print('Files in the results archive:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)
print('Archive:', archive_path)

try:
    from google.colab import files
except ImportError:
    print("Download the ZIP from the runtime file browser.")
else:
    files.download(archive_path)